# EDA
Run the pipeline first: `python -m src.data_pipeline`.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
df = pd.read_parquet(ROOT / "data/processed/features.parquet")
print(df.shape, df.date.min(), df.date.max())
df.head()


In [ ]:
df.isna().sum().sort_values(ascending=False).head(12)


In [ ]:
daily = df.groupby("date", observed=True)["sales"].sum()
daily.plot(figsize=(12,4), title="Total daily sales")
plt.show()


In [ ]:

import pandas as pd

df = pd.read_parquet(ROOT / "data/processed/features.parquet")

daily = (
    df.groupby("date")
    .agg(
        total_sales=("sales", "sum"),
        record_count=("sales", "size")
    )
    .reset_index()
)

# 10 ngày có doanh số thấp nhất
print(daily.nsmallest(10, "total_sales").to_string(index=False))

# Kiểm tra ngày hoàn toàn không có bản ghi
all_dates = pd.date_range(
    df["date"].min(), df["date"].max(), freq="D"
)
missing_dates = all_dates.difference(daily["date"])

print("\nMissing dates:", missing_dates.tolist())


      date   total_sales  record_count
2013-01-01   2511.618896          1782
2014-01-01   8602.065430          1782
2017-01-01  12082.501953          1782
2015-01-01  12773.617188          1782
2016-01-01  16433.394531          1782
2013-01-24 247245.687500          1782
2013-03-14 247541.406250          1782
2013-01-10 258982.000000          1782
2013-01-29 264488.812500          1782
2013-06-27 266525.437500          1782

Missing dates: [Timestamp('2013-12-25 00:00:00'), Timestamp('2014-12-25 00:00:00'), Timestamp('2015-12-25 00:00:00'), Timestamp('2016-12-25 00:00:00')]


In [5]:

# Liệt kê toàn bộ ngày không có bản ghi
print("=== ALL MISSING DATES ===")
for date in missing_dates:
    print(date.date())

# Kiểm tra số bản ghi có sales = 0
# trong ngày 01/01/2013
jan1 = df[df["date"] == "2013-01-01"]

print("\n=== JAN 1, 2013 ===")
print("Total records:", len(jan1))
print("Zero sales records:", (jan1["sales"] == 0).sum())
print("Positive sales records:", (jan1["sales"] > 0).sum())


=== ALL MISSING DATES ===
2013-12-25
2014-12-25
2015-12-25
2016-12-25

=== JAN 1, 2013 ===
Total records: 1782
Zero sales records: 1765
Positive sales records: 17


In [3]:
import pandas as pd

df = pd.read_parquet("../data/processed/features.parquet")

print(df.shape)
display(df.head())

(3000888, 19)


,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,year,month,day_of_week,day_of_month,is_weekend,sales_lag_1,sales_lag_7,sales_rolling_mean_7,sales_lag_14,sales_lag_28
0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,2013,1,1,1,0,NaN,NaN,NaN,NaN,NaN
1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,2013,1,1,1,0,NaN,NaN,NaN,NaN,NaN
2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,2013,1,1,1,0,NaN,NaN,NaN,NaN,NaN
3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,2013,1,1,1,0,NaN,NaN,NaN,NaN,NaN
4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,2013,1,1,1,0,NaN,NaN,NaN,NaN,NaN


In [4]:
sample = df.loc[
    (df["store_nbr"] == 1) &
    (df["family"] == "GROCERY I"),
    ["date", "sales", "sales_lag_7"]
]

sample = sample.sort_values("date").head(10)

display(sample)

,date,sales,sales_lag_7
12,2013-01-01,0.0,NaN
1794,2013-01-02,2652.0,NaN
3576,2013-01-03,2121.0,NaN
5358,2013-01-04,2056.0,NaN
7140,2013-01-05,2216.0,NaN
8922,2013-01-06,723.0,NaN
10704,2013-01-07,2262.0,NaN
12486,2013-01-08,1782.0,0.0
14268,2013-01-09,2429.0,2652.0
16050,2013-01-10,1544.0,2121.0


In [5]:

import pandas as pd
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent

sys.path.insert(0, str(root))

from src.baseline import predict_seasonal_naive, calculate_metrics

example = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=10),
    "store_nbr": [1] * 10,
    "family": ["GROCERY"] * 10,
    "sales": [100, 120, 140, 160, 180, 200, 220, 110, 130, 150]
})

result = predict_seasonal_naive(
    example,
    cutoff="2026-01-07",
    horizon=3
)

display(result[["date", "reference_date", "sales", "prediction"]])

print(calculate_metrics(result))


,date,reference_date,sales,prediction
0,2026-01-08,2026-01-01,110,100
1,2026-01-09,2026-01-02,130,120
2,2026-01-10,2026-01-03,150,140


{'observed_targets': 3, 'scored_targets': 3, 'coverage': 1.0, 'mae': 10.0, 'rmsle': 0.08147312012997475}


In [6]:

from src.train_model import prepare_data, build_origin_examples

# Chuẩn bị dữ liệu
prepared = prepare_data(df)

# Tạo dữ liệu phục vụ dự báo
examples = build_origin_examples(
    prepared,
    cutoff="2017-08-08",
    horizon=3
)

# Chỉ xem cửa hàng 1, một nhóm sản phẩm
sample = examples.loc[
    examples["store_nbr"] == 1
].head(5)

display(sample[[
    "date",
    "store_nbr",
    "family_code",
    "sales",
    "horizon_step",
    "baseline_reference",
    "cutoff_sales"
]])


,date,store_nbr,family_code,sales,horizon_step,baseline_reference,cutoff_sales
1171,2017-08-09,1,23,2.0,1,7.0,9.0
1172,2017-08-09,1,22,99.0,1,81.0,87.0
1173,2017-08-09,1,21,0.0,1,10.0,16.0
1174,2017-08-09,1,20,20.0,1,31.0,19.0
1175,2017-08-09,1,19,31.0,1,18.0,19.0


In [12]:

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(42)

# Tạo dữ liệu mẫu
n = 300
dates = pd.date_range("2025-01-01", periods=n)

weekday = dates.dayofweek

sales = (
    200
    + 30 * (weekday >= 5)
    + 15 * np.sin(2 * np.pi * np.arange(n) / 30)
    + rng.normal(0, 10, n)
)

data = pd.DataFrame({
    "date": dates,
    "day_of_week": weekday,
    "is_weekend": (weekday >= 5).astype(int),
    "day_of_month": dates.day,
    "sales": sales
})

features = ["day_of_week", "is_weekend", "day_of_month"]

# Chia dữ liệu theo thời gian
train = data.iloc[:200]
validation = data.iloc[200:250]
test = data.iloc[250:]

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))


Train: 200
Validation: 50
Test: 50


C:\Users\Admin\AppData\Local\Temp\ipykernel_19332\3656795170.py:13: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  weekday = dates.dayofweek


In [13]:

configs = [
    {"max_iter": 50, "learning_rate": 0.05},
    {"max_iter": 100, "learning_rate": 0.05},
    {"max_iter": 200, "learning_rate": 0.05}
]

results = []

for config in configs:

    model = HistGradientBoostingRegressor(
        **config,
        max_leaf_nodes=15,
        min_samples_leaf=10,
        random_state=42
    )

    model.fit(train[features], train["sales"])

    train_pred = model.predict(train[features])
    val_pred = model.predict(validation[features])

    results.append({
        **config,
        "train_mae": mean_absolute_error(
            train["sales"], train_pred
        ),
        "val_mae": mean_absolute_error(
            validation["sales"], val_pred
        )
    })

results_df = pd.DataFrame(results)

display(results_df.round(2))


,max_iter,learning_rate,train_mae,val_mae
0,50,0.05,6.41,9.89
1,100,0.05,5.84,9.86
2,200,0.05,5.31,9.88


In [15]:
best_config = min(
    results,
    key=lambda x: x["val_mae"]
)

print("Best configuration:")
print(best_config)

Best configuration:
{'max_iter': 100, 'learning_rate': 0.05, 'train_mae': 5.839090595660998, 'val_mae': 9.86281588754542}


In [16]:

train_val = pd.concat(
    [train, validation],
    ignore_index=True
)

final_model = HistGradientBoostingRegressor(
    max_iter=best_config["max_iter"],
    learning_rate=best_config["learning_rate"],
    max_leaf_nodes=15,
    min_samples_leaf=10,
    random_state=42
)

final_model.fit(
    train_val[features],
    train_val["sales"]
)

test_pred = final_model.predict(test[features])

test_mae = mean_absolute_error(
    test["sales"],
    test_pred
)

print(f"Final Test MAE: {test_mae:.2f}")


Final Test MAE: 9.43


## Your observations
1. ...
2. ...
3. ...

**Caution:** Historical lags inside a holdout must not be used naively for a fixed-origin 14-day forecast.
